# Subreddit groups (v4 sample, pre-accusation subreddits)

One group per subreddit, for the time-varying AFT variables. Rule-based groups take priority, in this order:

1. `profile` — user-profile subreddits `u_<name>` (own vs someone else's profile is decided per action, not here);
2. `banned` — PRAW returned no metadata and the subreddit is public/restricted/archived (as of the 2026-10 lookup);
3. `private` — PRAW returned no metadata and the subreddit is private;
4. `adult` — live subreddit labelled NSFW by Reddit (`over18`).

Every other subreddit (live, not NSFW) was classified from its title, bio and sidebar — falling back to its name only when
those were empty or uninformative — into crypto, politics, karma_farming, general, news, sales, location or other,
following `subreddit_classification_guide.md`. That classification was done by Claude agents reading each entry (2026-10-06);
labels, source (bio/name), confidence and a short reason are in `subreddit_review_output/subreddit_llm_labels_v4.csv`.

Inputs: `subreddit_review_output/subreddit_praw_meta_v4.parquet` (from `subreddit_metadata_praw_sample_v4.ipynb`) and the label file.
Outputs (never overwritten): `subreddit_groups_v4.parquet` and a review spreadsheet in `subreddit_review_output/`.

In [ ]:
# Libraries, folders, and the groups the labelling agents could choose from.
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

SAMPLE_DIR = 'sample_1000_v4_output'
OUT_DIR = 'subreddit_review_output'
LLM_GROUPS = ['crypto', 'politics', 'karma_farming', 'general', 'news', 'sales', 'location', 'other']

# Write a file only if it does not exist yet (never overwrite); .xlsx goes to Excel, anything else to parquet.
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    if name.endswith('.xlsx'):
        df.to_excel(path, index=False, freeze_panes=(1, 1))
    else:
        df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

In [ ]:
# PRAW metadata for each subreddit, and the agent labels for subreddits no rule covers.
meta = pd.read_parquet(os.path.join(OUT_DIR, 'subreddit_praw_meta_v4.parquet'))
llm = pd.read_csv(os.path.join(OUT_DIR, 'subreddit_llm_labels_v4.csv')).drop(columns='batch')
assert llm['subreddit'].is_unique and set(llm['group']) <= set(LLM_GROUPS)

# Rule-based groups first: profile, banned, private, adult (live NSFW flag).
is_profile = meta['subreddit'].str.startswith('u_')
no_meta = meta['status'].str.startswith('banned')          # PRAW returned no metadata
groups = meta[['subreddit', 'subreddit_id', 'status', 'subreddit_type', 'over18', 'title', 'public_description']].copy()
groups['group'] = np.select(
    [is_profile,
     no_meta & meta['subreddit_type'].isin(['public', 'restricted', 'archived']),
     no_meta & meta['subreddit_type'].eq('private'),
     meta['status'].eq('ok') & meta['over18'].fillna(False).astype(bool)],
    ['profile', 'banned', 'private', 'adult'], default='')
groups['group_source'] = np.where(groups['group'].ne(''), 'rule', '')
# Every other subreddit takes its agent label; checks that rules and labels do not overlap.
groups = groups.merge(llm.rename(columns={'group': 'llm_group'}), on='subreddit', how='left', validate='one_to_one')
need = groups['group'].eq('')
assert groups.loc[need, 'llm_group'].notna().all(), 'a subreddit needing a label has none'
assert groups.loc[~need, 'llm_group'].isna().all(), 'a rule-based subreddit was also sent for labelling'
groups.loc[need, 'group'] = groups.loc[need, 'llm_group']
groups.loc[need, 'group_source'] = 'llm_' + groups.loc[need, 'source']
groups = groups.drop(columns=['llm_group', 'source'])
print(groups['group'].value_counts().to_string())

In [ ]:
# Every post and comment the sample made before its first (edit-adjusted) accusation, tagged with its subreddit's group.
accounts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet')).set_index('username_key')
acts = []
for f in ['sample_posts.parquet', 'sample_comments.parquet']:
    d = pd.read_parquet(os.path.join(SAMPLE_DIR, f), columns=['outer_key', 'created_utc', 'subreddit'])
    acts.append(d[d['created_utc'] < d['outer_key'].map(accounts['first_mention_adj_utc'])])
acts = pd.concat(acts)
use = acts.groupby('subreddit').agg(actions=('outer_key', 'size'), accounts=('outer_key', 'nunique'))
groups = groups.join(use, on='subreddit')
acts['group'] = acts['subreddit'].map(groups.set_index('subreddit')['group'])
assert acts['group'].notna().all()
# Per group: subreddits, actions and accounts, and the share of all pre-accusation actions.
summary = pd.DataFrame({'subreddits': groups['group'].value_counts(), 'actions': acts['group'].value_counts(),
                        'accounts': acts.groupby('group')['outer_key'].nunique()})
summary['pct_subreddits'] = (100 * summary['subreddits'] / summary['subreddits'].sum()).round(1)
summary['pct_actions'] = (100 * summary['actions'] / summary['actions'].sum()).round(1)
summary.sort_values('actions', ascending=False)

In [ ]:
# Save the groups, and a review sheet sorted by group and use.
save_new(groups, 'subreddit_groups_v4.parquet')
review = groups.sort_values(['group', 'actions'], ascending=[True, False])[
    ['subreddit', 'group', 'group_source', 'confidence', 'reason', 'actions', 'accounts', 'title', 'public_description', 'status']]
save_new(review, 'subreddit_groups_v4_review.xlsx')